# Test File Loader (Final Orchestration)
This notebook validates the full `FileLoader` orchestration, including data encoding, constraint encoding, and violation detection.

## 1. Orchestration & Encoding Test

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.loading.file_loader import FileLoader
from src.loading.components.data_loader import DataLoader
from src.loading.components.dcs_loader import DCsLoader
from src.loading.components.metadata_loader import MetadataLoader
from src.loading.components.data_encoder import DataEncoder
from src.loading.components.dcs_encoder import DCsEncoder
import pandas as pd

loader = FileLoader(
    name="dummy", 
    base_path="../data",
    data_loader=DataLoader(),
    dcs_loader=DCsLoader(),
    metadata_loader=MetadataLoader(),
    data_encoder=DataEncoder(),
    dcs_encoder=DCsEncoder()
)

dataset = loader.load()

print("Encoded Data:")
display(dataset.data)
print(f"Data Dtypes:\n{dataset.data.dtypes}")

print("\nEncoded Constraints:")
for dc in dataset.dcs.constraints:
    print(dc)

assert all(pd.api.types.is_numeric_dtype(dataset.data[col]) for col in dataset.data.columns)
# Check if 'NYC' (City) was encoded to a number in constraints
city_dc = dataset.dcs.constraints[0]
city_predicate = city_dc.predicates[0]
import numpy as np
assert isinstance(city_predicate.right.attr, (int, float, np.number)), f"Expected numeric attribute, got {type(city_predicate.right.attr)}"
print("\nLogical tests passed!")

## 2. Violation Detection

In [ ]:
violations = dataset.violations
print(f"Violations found: {len(violations)}")
display(violations)

# NYC rows are 0 and 2. 
# DC1: t1.City = 'NYC' -> Row 0 and 2 violate this.
# DC2: t1.Salary > 4000 -> All 3 rows violate this.
assert len(violations) >= 3